In [ ]:
import numpy as np 
import pandas as pd 
import math
import scipy
import re
import os
import ast
import csv
import subprocess
import tempfile
import random
from collections import OrderedDict
from collections import Counter

# Plot 
import matplotlib
from matplotlib import pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns 
from scipy.cluster import hierarchy
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import networkx as nx
import PIL.Image as Image 
from netgraph import Graph
import plotly.graph_objects as go
import kaleido

# Interpretable Feature Calculation
from Bio.SeqUtils import MeltingTemp as mt
from scipy.stats import entropy
from scipy.stats import chi2_contingency
from scipy.stats import ks_2samp

# Correlation Calculation
from scipy.stats import spearmanr
from scipy.stats import pearsonr

# RNA-FM 
import torch
import fm
from tqdm.notebook import tqdm

In [ ]:
random.seed(42)

# 1. Dataset Initialisation

In [ ]:
# Data preparation for embeddings
def prepare_emb(df):
    data = []
    for idx in df.index:
        if 'Sequence' in df.columns:
            data.append(tuple((idx, df.loc[idx,'Sequence'])))
        else:
            data.append(tuple((idx, df.loc[idx,'sequence'])))

    print(data[0])
    print(len(data))
    
    return data

In [ ]:
# Data files containing sequences 
mirna = pd.read_csv('mirna_sequences_whole.csv', index_col=0)
sirna = pd.read_csv('sirna_sequences_whole.csv', index_col=0)
pirna = pd.read_csv('pirna_sequences_whole.csv', index_col=0)

print(f'Total Number of Sequences: {len(mirna) + len(sirna) + len(pirna)}')

mirna['length'] = mirna['sequence'].apply(len)
sirna['length'] = sirna['sequence'].apply(len)
pirna['length'] = pirna['sequence'].apply(len)

# For unique sequences
mirna = mirna.drop_duplicates(['sequence']).reset_index(drop=True)
sirna = sirna.drop_duplicates(['sequence']).reset_index(drop=True)
pirna = pirna.drop_duplicates(['sequence']).reset_index(drop=True)

print(f'Total Number of Sequences: {len(mirna) + len(sirna) + len(pirna)}')

In [ ]:
# To prepare input data
# for Data-Whole
# mirna_data = prepare_emb(mirna)
# sirna_data = prepare_emb(sirna)
# pirna_data = prepare_emb(pirna)

# to generate random sequences 
# for Data-23K 
# mirna_data = random.sample(mirna_data, k=len(pirna_data)) 
# sirna_data = random.sample(sirna_data, k=len(pirna_data))

# for Data-3K 
# sirna_data = random.sample(sirna_data, k=len(mirna_data))
# pirna_data = random.sample(pirna_data, k=len(mirna_data))

In [ ]:
# To use the dataset used in the paper 
# with open("feat_emb_correlation/unique/whole/mirna_embeddings.txt", 'r') as file: # Data-Whole
with open("feat_emb_correlation/unique/23k/mirna_embeddings_23k.txt", 'r') as file: # Data-23K 
# with open("feat_emb_correlation/unique/3k/mirna_embeddings_3k.txt", 'r') as file: # Data-3K 
    mirna_data = []
    for line in file:
        key, val = line.split(',')
        key = int(key.replace('(',''))
        val = re.sub(r'[^a-zA-Z]', '', val)
        mirna_data.append(tuple([key, val]))
    print(mirna_data[0])
    print(len(mirna_data))


# with open("feat_emb_correlation/unique/whole/sirna_embeddings.txt", 'r') as file: # Data-Whole
with open("feat_emb_correlation/unique/23k/sirna_embeddings_23k.txt", 'r') as file: # Data-23K 
# with open("feat_emb_correlation/unique/3k/sirna_embeddings_3k.txt", 'r') as file: # Data-3K 
    sirna_data = []
    for line in file:
        key, val = line.split(',')
        key = int(key.replace('(',''))
        val = re.sub(r'[^a-zA-Z]', '', val)
        sirna_data.append(tuple([key, val]))
    print(sirna_data[0])
    print(len(sirna_data))


with open("feat_emb_correlation/unique/whole/pirna_embeddings.txt", 'r') as file: # Data-Whole and Data-23K
# with open("feat_emb_correlation/unique/3k/pirna_embeddings_3k.txt", 'r') as file: # Data-3K 
    pirna_data = []
    for line in file:
        key, val = line.split(',')
        key = int(key.replace('(',''))
        val = re.sub(r'[^a-zA-Z]', '', val)
        pirna_data.append(tuple([key, val]))
    print(pirna_data[0])
    print(len(pirna_data))

# 2. Feature Generation

In [ ]:
# If regenerating from scratch 
def calculate_rna_thermodynamics(sequence):
    result = subprocess.run(
        ['RNAfold', '-p', '-d2', '--noLP'], 
        input=sequence, capture_output=True, text=True
    )
    output = result.stdout.split("\n")  # Split output into lines
    mfe, gibbs_free_energy = None, None

    for line in output:
        mfe_match = re.search(r"\(\s*(-?\d+\.\d+)\s*\)", line)
        if mfe_match:
            mfe = float(mfe_match.group(1))  # Extract MFE

        # Extract Gibbs Free Energy from partition function output
        gibbs_match = re.search(r"\[\s?([-]?\d+\.\d+)\s?\]", line)
        if gibbs_match:
            gibbs_free_energy = float(gibbs_match.group(1))  # Extract Gibbs Free Energy

    # The Gibbs Free Energy (ΔG) values for RNA secondary structures were computed using RNAfold's partition function approach (-p option), 
    # which calculates the ensemble free energy based on the Boltzmann-weighted sum of all possible conformations. 
    # The reported ΔG represents the thermodynamic stability of the RNA molecule at 37°C
    
    return mfe, gibbs_free_energy

def run_rnafold(sequence):
    with tempfile.NamedTemporaryFile(mode="w+", delete=False) as temp_file:
        temp_file.write(sequence)
        temp_file.flush()
        result = subprocess.run(["RNAfold", "--noPS"], stdin=open(temp_file.name), stdout=subprocess.PIPE, text=True)
        os.unlink(temp_file.name)
        output = result.stdout.strip().split('\n')
        if len(output) >= 2:
            structure = output[1].split()[0]
            return structure
        return None

# Function to find repeats and palindromes
def find_repeats_and_palindromes(sequence, min_len=4):
    palindromes = []
    for i in range(len(sequence)):
        for j in range(i + min_len, len(sequence) + 1):
            substring = sequence[i:j]
            if substring == substring[::-1]:
                palindromes.append(substring)
    return len(palindromes), palindromes

# Function to calculate Melting Temperature (Tm)
def calculate_tm(sequence):
    return mt.Tm_NN(sequence, nn_table=mt.RNA_NN1)

# Function to calculate nucleotide skew metrics
def calculate_skew(sequence):
    a = sequence.count('A')
    u = sequence.count('U')
    g = sequence.count('G')
    c = sequence.count('C')
    au_skew = (a - u) / (a + u) if (a + u) > 0 else 0
    gc_skew = (g - c) / (g + c) if (g + c) > 0 else 0
    return au_skew, gc_skew

def calculate_shannon_entropy(sequence):
    count = Counter(sequence)
    probabilities = np.array(list(count.values())) / len(sequence)
    return entropy(probabilities, base=2)  # base 2 for bits

def nucleotide_distribution(sequences):
    nucleotides = ['A', 'C', 'G', 'U']
    counts = Counter(''.join(sequences))
    total_count = sum(counts.values())
    return {nuc: counts[nuc] / total_count for nuc in nucleotides}

def calculate_gc_content(sequence):
    return round((sequence.count('G') + sequence.count('C')) / len(sequence), 3)

def generate_pos_feat(sequence):
    # Reference: Bai,Y., Zhong,H., Wang,T. and Lu,Z.J. (2024) OligoFormer: An accurate and robust prediction method for siRNA design. Bioinformatics, 10.1093/bioinformatics/btae577. 
    pos_feat = {
        'G_1': 1 if sequence[0] == 'G' else 0,
        'U_1': 1 if sequence[0] == 'U' else 0,
        'GG_all': [sequence[j]+sequence[j+1] for j in range(len(sequence)-1)].count('GG') / (len(sequence)-1), 
        'UA_all': [sequence[j]+sequence[j+1] for j in range(len(sequence)-1)].count('UA') / (len(sequence)-1),
        'CC_all': [sequence[j]+sequence[j+1] for j in range(len(sequence)-1)].count('CC') / (len(sequence)-1),
        'GC_all': [sequence[j]+sequence[j+1] for j in range(len(sequence)-1)].count('GC') / (len(sequence)-1),
        'UU_all': [sequence[j]+sequence[j+1] for j in range(len(sequence)-1)].count('UU') / (len(sequence)-1)
    }

    return pos_feat

def process_rna_sequences(input_sequences, output_csv):
    # Load sequences from CSV
    df = pd.DataFrame(input_sequences, columns=['id', 'sequence'])
    sequences = df['sequence']

    # Prepare lists to store calculated features
    results = {'sequence': [],
                'gc_content':  [],
                'gibbs_free_energy':  [],
                'minimum_free_energy': [],
                'a_content':  [],
                'u_content':  [], 
                'g_content': [],
                'c_content': [],
                'num_palindromes':  [],
                'melting_temp':  [],
                'au_skew':  [],
                'gc_skew':  [],
                'shannon_entropy':  [],
                'U_1':  [],
                'G_1':  [],
                'GG_all':  [],
                'UA_all':  [],
                'CC_all':  [],
                'GC_all':  [],
                'UU_all':  [],
                'secondary_structure': []}
    
    # Process each sequence
    for sequence in sequences:
        sequence = sequence.strip()
        results['sequence'].append(sequence)

        # GC Content
        try:
            gc_content = calculate_gc_content(sequence)
        except Exception:
            gc_content = None
        results['gc_content'].append(gc_content)

        # MFE and structure
        try:
            mfe, gibbs_free_energy = calculate_rna_thermodynamics(sequence)
        except Exception:
            mfe, gibbs_free_energy = None, None
        results['gibbs_free_energy'].append(gibbs_free_energy)
        results['minimum_free_energy'].append(mfe)

        # # Repeats and Palindromes
        # try:
        #     num_palindromes, palindromes = find_repeats_and_palindromes(sequence)
        # except Exception:
        #     num_palindromes, palindromes = None, []
        # results['num_palindromes'].append(num_palindromes)
        # results['palindromes'].append(", ".join(palindromes))

        # Melting Temperature (Tm)
        try:
            tm = calculate_tm(sequence)
        except Exception:
            tm = None
        results['melting_temp'].append(tm)

        try:
            sec_struct = run_rnafold(sequence)
        except Exception:
            sec_struct = None
        results['secondary_structure'].append(sec_struct)

        # Skew Metrics
        try:
            au_skew, gc_skew = calculate_skew(sequence)
        except Exception:
            au_skew, gc_skew = None, None
        results['au_skew'].append(au_skew)
        results['gc_skew'].append(gc_skew)

        # Shannon Entropy
        try:
            entropy = calculate_shannon_entropy(sequence)
        except Exception:
            entropy = None
        results['shannon_entropy'].append(entropy)

        # Nucleotide Distribution
        try:
            dist = nucleotide_distribution([sequence])
        except Exception:
            dist = None
        for nuc in dist.keys():
            results[f'{nuc.lower()}_content'].append(dist[nuc])

        # Positonal features
        try: 
            pos_feat = generate_pos_feat(sequence)
        except Exception:
            pos_feat = {'G_1': None, 'U_1': None, 'GG_all': None, 'UA_all': None, 'CC_all': None, 'GC_all': None, 'UU_all': None}
        for pos_key in pos_feat:
            results[pos_key].append(pos_feat[pos_key])


    for key, value in results.items():
        if len(value) < len(sequences):
            value.extend([None] * (len(sequences) - len(value)))

    # Creating a DataFrame and save it to CSV
    output_df = pd.DataFrame(results)
    output_df.to_csv(output_csv, index=False)
    print(f"Features saved to {output_csv}")

    return output_df


sirna = process_rna_sequences(sirna_data, './Dataset/features/siRNA_features.csv')
mirna = process_rna_sequences(mirna_data, './Dataset/features/miRNA_features.csv')
pirna = process_rna_sequences(pirna_data, './Dataset/features/piRNA_features.csv')

# 3. Interpretable Feature Analysis

In [ ]:
# Density distribution plot with KS test 
def detailedFeatureBoxenGrid(siRNA, miRNA, piRNA, output_file_path, dataNum):
    siRNA = siRNA.copy()
    miRNA = miRNA.copy()
    piRNA = piRNA.copy() 
    
    siRNA['RNA_type'] = 'siRNA'
    miRNA['RNA_type'] = 'miRNA'
    piRNA['RNA_type'] = 'piRNA'

    data = pd.concat([siRNA, miRNA, piRNA], ignore_index=True)

    rename_dict = {
        'gibbs_free_energy': 'Gibbs Free Energy',
        'minimum_free_energy': 'Minimum Free Energy',
        'melting_temp': 'Melting Temperature',
        'shannon_entropy': 'Shannon Entropy',
        'gc_content': 'GC Content',
        'au_skew': 'AU Skew',
        'gc_skew': 'GC Skew',
        'a_content': 'Nucleotide A',
        'c_content': 'Nucleotide C',
        'g_content': 'Nucleotide G',
        'u_content': 'Nucleotide U',
        'GG_all': 'GG across Sequence',
        'UA_all': 'UA across Sequence',
        'CC_all': 'CC across Sequence',
        'GC_all': 'GC across Sequence',
        'UU_all': 'UU across Sequence'
    }
    data = data.rename(columns=rename_dict)

    features_to_plot = list(rename_dict.values())
    num_features = len(features_to_plot)

    cols = 3
    rows = math.ceil(num_features / cols)
    colors = {"siRNA": "#1f77b4", "miRNA": "#ff7f0e", "piRNA": "#2ca02c"}

    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
    axes = axes.flatten()

    def p_to_stars(p):
        if p < 1e-4: return "****"
        if p < 1e-3: return "***"
        if p < 1e-2: return "**"
        if p < 5e-2: return "*"
        return "ns"

    def annotate_comparison(ax, x1, x2, y, text, h=0.02):
        ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=1, c='k')
        ax.text((x1 + x2) / 2, y + h, text, ha='center', va='bottom', fontsize=10)

    order = ["siRNA", "miRNA", "piRNA"]
    pairs = [(0,1,"siRNA","miRNA"), (0,2,"siRNA","piRNA"), (1,2,"miRNA","piRNA")]

    for idx, feature in enumerate(features_to_plot):
        ax = axes[idx]

        sns.boxenplot(
            x="RNA_type", y=feature, data=data, order=order,
            palette=colors, ax=ax
        )
        ax.set_title(f"{feature} Distribution", fontsize=12)
        ax.set_xlabel("RNA Type")
        ax.set_ylabel(feature)

        # compute pairwise KS tests on non-NaN values
        y_max = data[feature].dropna().max()
        y_min = data[feature].dropna().min()
        y_range = (y_max - y_min) if np.isfinite(y_max) and np.isfinite(y_min) else 1.0
        base = y_max if np.isfinite(y_max) else 1.0

        step = 0.06 * y_range if y_range > 0 else 1.0
        cur_y = base + 0.05 * y_range

        for j, (x1, x2, g1, g2) in enumerate(pairs):
            grp1 = data.loc[data["RNA_type"] == g1, feature].dropna().values
            grp2 = data.loc[data["RNA_type"] == g2, feature].dropna().values
            if len(grp1) >= 2 and len(grp2) >= 2:
                stat, p = ks_2samp(grp1, grp2, alternative='two-sided', mode='auto')
                stars = p_to_stars(p)
                # annotate_comparison(ax, x1, x2, cur_y, stars, h=0.01 * y_range)
                cur_y += step

        ax.set_ylim(ax.get_ylim()[0], max(ax.get_ylim()[1], cur_y + 0.05 * y_range))

    for j in range(idx + 1, len(axes)):
        axes[j].axis('off')

    plt.rcParams.update({"svg.fonttype": "none"})
    plt.tight_layout()
    os.makedirs(output_file_path, exist_ok=True)
    outpath = f'{output_file_path}/RNA_Feature_Distributions_ks_test_{dataNum}k.svg'
    plt.savefig(outpath, format='svg')
    plt.close()
    print(f"Saved: {outpath}")

detailedFeatureBoxenGrid(
    sirna,
    mirna, 
    pirna,
    "./Dataset/Figures", 
    "23"
)


In [ ]:
def detailedFeatureAnalysis(siRNA, miRNA, piRNA, output_file_path, dataNum):
    plt.rcParams.update({
        "svg.fonttype": "none",
    })
    
    siRNA = siRNA.copy()
    miRNA = miRNA.copy()
    piRNA = piRNA.copy()

    # Add a column for length 
    siRNA['Length'] = siRNA['sequence'].apply(len)
    miRNA['Length'] = miRNA['sequence'].apply(len)
    piRNA['Length'] = piRNA['sequence'].apply(len)

    # Add a column for RNA type
    siRNA['RNA_type'] = 'siRNA'
    miRNA['RNA_type'] = 'miRNA'
    piRNA['RNA_type'] = 'piRNA'

    data = pd.concat([siRNA, miRNA, piRNA], ignore_index=True)

    data = data.rename(columns={'a_content': 'A_distribution',
                                'g_content': 'G_distribution',
                                'u_content': 'U_distribution',
                                'c_content': 'C_distribution'})

    data = data.rename(columns={
        'gibbs_free_energy': 'Gibbs Free Energy',
        'minimum_free_energy': 'Minimum Free Energy',
        'melting_temp': 'Melting Temperature',
        'shannon_entropy': 'Shannon Entropy',
        'gc_content': 'GC Content',
        'au_skew': 'AU Skew',
        'gc_skew': 'GC Skew',
        'A_distribution': 'Nucleotide A', 
        'C_distribution': 'Nucleotide C', 
        'G_distribution': 'Nucleotide G', 
        'U_distribution': 'Nucleotide U',
        'U_1': 'U at Position 1', 
        'G_1': 'G at Position 1', 
        'GG_all': 'GG across Sequence', 
        'UA_all': 'UA across Sequence', 
        'CC_all': 'CC across Sequence', 
        'GC_all': 'GC across Sequence', 
        'UU_all': 'UU across Sequence',
        'length': 'Length',
    })
 
    results2 = ['Gibbs Free Energy','Minimum Free Energy','Melting Temperature','Shannon Entropy', 'GC Content', 'AU Skew','GC Skew', 'Nucleotide A', 'Nucleotide C', 'Nucleotide G', 'Nucleotide U', 'U at Position 1', 'G at Position 1', 'GG across Sequence', 'UA across Sequence', 'CC across Sequence', 'GC across Sequence', 'UU across Sequence', 'Length']
    
    # Define colors for RNA types
    colors = {"siRNA": "#1f77b4", "miRNA": "#ff7f0e", "piRNA": "#2ca02c"}

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Boxplot for Gibbs Free Energy
    sns.boxplot(data=data, x='RNA_type', y='Gibbs Free Energy', palette=colors, ax=axes[0, 0])
    axes[0, 0].set_title('Gibbs Free Energy Distribution')
    axes[0, 0].set_xlabel('RNA Type')
    axes[0, 0].set_ylabel('Gibbs Free Energy')

    # Boxplot for Minimum Free Energy
    sns.boxplot(data=data, x='RNA_type', y='Minimum Free Energy', palette=colors, ax=axes[0, 1])
    axes[0, 1].set_title('Minimum Free Energy Distribution')
    axes[0, 1].set_xlabel('RNA Type')
    axes[0, 1].set_ylabel('Minimum Free Energy')

    # Boxplot for Melting Temperature
    sns.boxplot(data=data, x='RNA_type', y='Melting Temperature', palette=colors, ax=axes[1, 0])
    axes[1, 0].set_title('Melting Temperature Distribution')
    axes[1, 0].set_xlabel('RNA Type')
    axes[1, 0].set_ylabel('Melting Temperature')

    # Boxplot for Shannon Entropy
    sns.boxplot(data=data, x='RNA_type', y='Shannon Entropy', palette=colors, ax=axes[1, 1])
    axes[1, 1].set_title('Shannon Entropy Distribution')
    axes[1, 1].set_xlabel('RNA Type')
    axes[1, 1].set_ylabel('Shannon Entropy')

    # Adjust layout for better spacing
    plt.tight_layout()
    output_file = f'{output_file_path}/gibbs_free_energy_min_free_energy_melting_temp_Shannon_entropy_for_{dataNum}k.svg'
    plt.savefig(output_file, format='svg')

    # Correlation Heatmap
    plt.figure(figsize=(14, 6))
    df1 = data.copy()
    corr = df1[results2].corr()
    sns.heatmap(corr, annot=False, cmap='coolwarm', fmt='.2f', cbar_kws={'label': "Pearson's Correlation Coefficient"})
    plt.title('Correlation Heatmap')
    output_file = f'{output_file_path}/correlation_heatmap_for_RNA_analysis_{dataNum}k.svg'
    plt.savefig(output_file, format='svg', bbox_inches='tight')
    plt.show()

    # Hexbin Plot
    window_size = 50
    df1 = data.copy()
    df1["gc_skew_smooth"] = df1.groupby("RNA_type")["GC Skew"].transform(lambda x: x.rolling(window=window_size, min_periods=1).mean())
    df1["au_skew_smooth"] = df1.groupby("RNA_type")["AU Skew"].transform(lambda x: x.rolling(window=window_size, min_periods=1).mean())
    plt.figure(figsize=(8, 6))
    hb = plt.hexbin(df1["gc_skew_smooth"], df1["au_skew_smooth"], gridsize=50, cmap="coolwarm", mincnt=1)
    plt.colorbar(hb, label="Count of Sequences")
    plt.xlabel("GC Skew")
    plt.ylabel("AU Skew")
    plt.title("GC Skew vs AU Skew")
    plt.savefig(f'{output_file_path}/hexbin_gc_au_skew_{dataNum}k.svg', format="svg", dpi=600)

    # Violin Plot
    fig, box_axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True)
    sns.boxenplot(x="RNA_type", y="au_skew_smooth", data=df1, palette=colors, ax=box_axes[0])
    box_axes[0].set_title("AU Skew Distribution")
    box_axes[0].set_xlabel("RNA Type")
    box_axes[0].set_ylabel("AU Skew")
    sns.boxenplot(x="RNA_type", y="gc_skew_smooth", data=df1, palette=colors, ax=box_axes[1])
    box_axes[1].set_title("GC Skew Distribution")
    box_axes[1].set_xlabel("RNA Type")
    box_axes[1].set_ylabel("GC Skew")
    # Adjust layout & save
    plt.tight_layout()
    plt.savefig(f'{output_file_path}/au_gc_skew_analysis_{dataNum}k.svg', format='svg', dpi=600)
    plt.show()

    #  Nucleotide distribution
    datasets = {
        'siRNA': data[data["RNA_type"] == "siRNA"],
        'miRNA': data[data["RNA_type"] == "miRNA"],
        'piRNA': data[data["RNA_type"] == "piRNA"]
    }
    nucleotide_correlation_matrices = {}
    nucleotide_features = ['Nucleotide A', 'Nucleotide C', 'Nucleotide G', 'Nucleotide U']

    for key, df in datasets.items():
        nucleotide_correlation_matrices[key] = df[nucleotide_features].corr()
        nucleotide_correlation_matrices[key].columns = ['A', 'C', 'G', 'U']
        nucleotide_correlation_matrices[key].index = ['A', 'C', 'G', 'U']

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for ax, (key, corr_matrix) in zip(axes, nucleotide_correlation_matrices.items()):
        sns.heatmap(corr_matrix, annot=True, cmap="viridis", fmt=".2f", ax=ax, vmin=-1, vmax=1)
        ax.set_title(f"Nucleotide Correlation for {key}")
        ax.set_xlabel("Nucleotides")
        ax.set_ylabel("Nucleotides")

    plt.tight_layout()
    # Save and display the figure
    output_file = f'{output_file_path}/Nucleotide_Correlation_Comparison_{dataNum}k.svg'
    plt.savefig(output_file, format='svg')

    # Structral motif
    def parse_structure(structure: str):
        stems  = structure.count("(") + structure.count(")")
        bulges = structure.count(".")
        return {"Stem": stems, "Bulge": bulges}

    def analyze_structures(df):
        motifs = {"Stem": 0, "Bulge": 0}
        for s in df['secondary_structure']:
            if isinstance(s, str) and len(s) > 0:
                m = parse_structure(s)
                motifs["Stem"]  += m["Stem"]
                motifs["Bulge"] += m["Bulge"]
        return motifs

    siRNA_motifs = analyze_structures(siRNA)
    miRNA_motifs = analyze_structures(miRNA)
    piRNA_motifs = analyze_structures(piRNA)

    motifs_df = pd.DataFrame(
        [siRNA_motifs, miRNA_motifs, piRNA_motifs],
        index=["siRNA","miRNA","piRNA"]
    )[["Stem","Bulge"]]

    totals = motifs_df.sum(axis=1)
    props  = (motifs_df.T / totals).T

    fig, ax = plt.subplots(figsize=(7,5))
    motifs = ["Stem", "Bulge"]
    x = np.arange(len(motifs))
    w = 0.25
    # Bar positions for the 3 classes
    pos = {
        "siRNA": x - w,
        "miRNA": x,
        "piRNA": x + w,
    }

    # Plot proportions
    bars = {}
    bars["siRNA"] = ax.bar(pos["siRNA"], props.loc["siRNA", motifs].values, width=w,
                        label="siRNA", color=colors["siRNA"], edgecolor="black", linewidth=0.5)
    bars["miRNA"] = ax.bar(pos["miRNA"], props.loc["miRNA", motifs].values, width=w,
                        label="miRNA", color=colors["miRNA"], edgecolor="black", linewidth=0.5)
    bars["piRNA"] = ax.bar(pos["piRNA"], props.loc["piRNA", motifs].values, width=w,
                        label="piRNA", color=colors["piRNA"], edgecolor="black", linewidth=0.5)

    # pairwise p-values per motif chi-square
    pairs = [("siRNA","miRNA"), ("siRNA","piRNA"), ("miRNA","piRNA")]
    pair_offsets = {("siRNA","miRNA"):-0.5*w, ("siRNA","piRNA"):0, ("miRNA","piRNA"):0.5*w}
    alpha = 0.05
    m_tests = len(pairs) * len(motifs)

    def bracket(ax, x1, x2, y, h=0.02, text="p="):
        """draw a bracket with text above"""
        ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1, c="black")
        ax.text((x1+x2)/2, y+h*1.1, text, ha="center", va="bottom", fontsize=9)
    
    def p_to_stars(p):
        if p < 0.001:
            return "***"
        elif p < 0.01:
            return "**"
        elif p <= 0.05:
            return "*"
        else:
            return "ns"   # not significant

    for j, m in enumerate(motifs):
        ymax = max(bars["siRNA"][j].get_height(),
                bars["miRNA"][j].get_height(),
                bars["piRNA"][j].get_height())
        base_y = ymax + 0.03
        bump = 0.05

        for k, (a,b) in enumerate(pairs):
            a_m = motifs_df.loc[a, m]; a_not = totals[a] - a_m
            b_m = motifs_df.loc[b, m]; b_not = totals[b] - b_m
            chi2, p, dof, _ = chi2_contingency([[a_m, a_not], [b_m, b_not]])
            p_adj = min(p * m_tests, 1.0)
            x1 = pos[a][j]; x2 = pos[b][j]
            x1 += pair_offsets[(a,b)]; x2 += pair_offsets[(a,b)]
            bracket(ax, x1, x2, base_y + k*bump,
                    text=(f"p={p_adj:.2e}" if p_adj >= 0.0001 else "p<0.0001"))
            # label = p_to_stars(p_adj)
            # label = f"{p_to_stars(p_adj)} (p={p_adj:.2e})"
            # bracket(ax, x1, x2, base_y + k*bump, text=label)

    plt.tight_layout()
    plt.savefig(f"{output_file_path}/Structural_Motif_Proportions_pairwiseP_{dataNum}k.svg",
                format="svg")
    
detailedFeatureAnalysis(sirna, mirna, pirna, './Dataset/Figures', '23')

In [ ]:
# Correlation Network graph 
fig, ax = plt.subplots(figsize=(16,13))

g = nx.Graph()

irrespective_network_df = pd.concat([sirna, mirna, pirna]).reset_index(drop=True)
irrespective_network_df = irrespective_network_df.drop(columns=['sequence', 'num_palindromes', 'secondary_structure'])
irrespective_network_df = irrespective_network_df.rename(columns={'gc_content': 'GC\nContent',
                                                                  'gibbs_free_energy':  'Gibbs\nFree\nEnergy',
                                                                    'minimum_free_energy': 'Minimum\nFree\nEnergy',
                                                                    'a_content':  'Nucleotide\nA',
                                                                    'u_content':  'Nucleotide\nU', 
                                                                    'g_content': 'Nucleotide\nG', 
                                                                    'c_content': 'Nucleotide\nC',
                                                                    'melting_temp':  'Melting\nTemperature',
                                                                    'au_skew':  'AU\nSkew',
                                                                    'gc_skew':  'GC\nSkew',
                                                                    'shannon_entropy':  'Shannon\nEntropy',
                                                                    'U_1': 'U\nat\nPosition\n1',
                                                                    'G_1':  'G\nat\nPosition\n1',
                                                                    'GG_all':  'GG\nacross\nSequence',
                                                                    'UA_all':  'UA\nacross\nSequence',
                                                                    'CC_all':  'CC\nacross\nSequence',
                                                                    'GC_all':  'GC\nacross\nSequence',
                                                                    'UU_all':  'UU\nacross\nSequence'})

network_dict = {}
for y in irrespective_network_df.corr(method='pearson'):
    network_dict[y] = sorted(irrespective_network_df.corr(method='pearson')[y].items(), key=lambda x: x[1])[:3] + sorted(irrespective_network_df.corr(method='pearson')[y].items(), key=lambda x: x[1], reverse=True)[:4]
    print(network_dict[y])
    g.add_node('\n'.join(y.split(' ')))
    for z in [x for x in network_dict[y]]:
        if z[0] != y: 
            g.add_node('\n'.join(z[0].split(' ')))
            g.add_edge('\n'.join(y.split(' ')), '\n'.join(z[0].split(' ')), weight=z[1])
            print(y, z[0], z[1])
print(g)

hex_colors = ['#B85F69', '#CB9098', '#9E577A']

colormap = []
for node,_ in g.nodes.items():
    if node in ['Nucleotide\nA', 'Nucleotide\nU', 'Nucleotide\nG', 'Nucleotide\nC', 'GC\nContent', 'GC\nSkew', 'AU\nSkew','Shannon\nEntropy']: #Structural 
        colormap.append(hex_colors[0])#[4])
    elif node in ['U\nat\nPosition\n1', 'UU\nacross\nSequence','G\nat\nPosition\n1', 'GG\nacross\nSequence', 'CC\nacross\nSequence', 'GC\nacross\nSequence', 'UA\nacross\nSequence']: # Psotional 
        colormap.append(hex_colors[1])#[0])
    elif node in ['Gibbs\nFree\nEnergy', 'Minimum\nFree\nEnergy', 'Melting\nTemperature']: #Thermo
        colormap.append(hex_colors[2])#[2]) 

edges, weights = zip(*nx.get_edge_attributes(g,'weight').items())
d = dict(g.degree)
nx.set_edge_attributes(g,values={k: abs(v) for k, v in nx.get_edge_attributes(g,'weight').items()}, name='weight') # such that weights are positive for kamada_kawai, but colours will represent neg/pos correlation
paths = dict(nx.shortest_path_length(g, weight=weights))
for x in paths:
    for y in paths[x]:
        paths[x][y] = paths[x][y]*2
pos = nx.kamada_kawai_layout(g,paths,scale=1)#shell_layout(g)

colourmap = sns.color_palette("coolwarm", as_cmap=True) 
norm = plt.Normalize(vmin = min(weights), vmax = max(weights))
edge_color = colourmap(norm(weights))
edge_width = [x*20 for x in list(map(abs,weights))]

for loop in range(20):
    for nodex in g.nodes():
        for nodey in g.nodes():
            if (nodex != nodey):
                # if y distance is too small
                if(max(pos[nodex][1],pos[nodey][1])-min(pos[nodex][1],pos[nodey][1])<0.3):
                    # check if also x distance is too small
                    if((max(pos[nodex][0],pos[nodey][0])-min(pos[nodex][0],pos[nodey][0])< 0.3)):
                        print(nodex, nodey)
                        if(pos[nodex][1] < pos[nodey][1]):
                            pos[nodex][1] = pos[nodex][1]-0.3
                            pos[nodey][1] = pos[nodey][1]+0.3
                        else:
                            pos[nodex][1] = pos[nodex][1]+0.3
                            pos[nodey][1] = pos[nodey][1]-0.3

for y in pos:
    if y in ['Minimum\nFree\nEnergy']:
        pos[y] = pos[y] + [-0.2, 0]
    elif y in ['Nucleotide\nU']:
        pos[y] = pos[y] + [0.1,0.1]
    elif y in ['Nucleotide\nG']:
        pos[y] = pos[y] + [0, -0.4]
    elif y in ['Nucleotide\nA']:
        pos[y] = pos[y] + [-0.1, -0.2]
    elif y in ['U\nat\nPosition\n1']:
        pos[y] = pos[y] + [-0.1, 0.1]
    elif y in ['Melting\nTemperature']:
        pos[y] = pos[y] + [-0.05, 0]
    elif y in ['Nucleotide\nC']:
        pos[y] = pos[y] + [-0.05, -0.1]
    elif y in ['Gibbs\nFree\nEnergy']:
        pos[y] = pos[y] + [-0.05, 0.2]


connections = [] 
for pair in g.edges(): 
    for node in g.nodes():
        if node not in pair:
            if min([pos[pair[0]][0], pos[pair[1]][0]]) < pos[node][0] < max([pos[pair[0]][0], pos[pair[1]][0]]) and min([pos[pair[0]][1], pos[pair[1]][1]]) < pos[node][1] < max([pos[pair[0]][1], pos[pair[1]][1]]):
                print(pair, node, pos[pair[0]], pos[pair[1]], pos[node])
                connections.append('arc3,rad=-0.3')
                break
        else: 
            connections.append('arc3')
            break


for count, node in enumerate(g.nodes()):
    nx.draw_networkx_nodes(g, pos, nodelist = [node], node_color=colormap[count], node_size=5600, edgecolors='black')
    nx.draw_networkx_labels(g, pos, {node:node}, font_size=12, font_color='white')

for id, edge in enumerate(g.edges()):
    print(edge)
    nx.draw_networkx_edges(g, pos, edgelist=[edge],arrows=True, edge_color=edge_color[id], edge_cmap=sns.color_palette("coolwarm", as_cmap=True), width = edge_width[id], connectionstyle=connections[id])
    
legend_dict= {'Structural': hex_colors[0],
             'Positional': hex_colors[1],
             'Thermodynamic':hex_colors[2]}
handles = [Patch(facecolor=legend_dict[label], label=label) for label in legend_dict]
ax.legend(handles=handles, loc=[0.97, 0.73], title='Type of Features', fontsize=14, title_fontsize=16)
ax.axis('off')
sm = plt.cm.ScalarMappable(cmap=colourmap, norm=plt.Normalize(vmin = -1, vmax=1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.4)
cbar.ax.tick_params(labelsize=14)
cbar.set_label("Pearson Correlation Coefficient", fontsize=14)

plt.tight_layout()
# plt.savefig("./pcc_23k_irrespective_network_2_final_shannon_v1.png", dpi=300)
# plt.savefig("./pcc_23k_irrespective_network_2_final_shannon_v1.svg", dpi=600)
plt.show()

# 4. Deep Learning Embeddings Analysis

## 4.1 t-SNE Projection

In [ ]:
# Generate RNA-FM embeddings
def Featurizer(input_data, chunk_size):
    # Load RNA-FM
    model, alphabet = fm.pretrained.rna_fm_t12('./.cache/torch/hub/checkpoints/RNA-FM_pretrained.pth')
    emb_len = 640

    batch_converter = alphabet.get_batch_converter()
    model.eval()  # disables dropout for deterministic results

    embeddings = np.zeros((len(input_data), emb_len))

    # Prepare data
    for i in tqdm(range(0, len(input_data), chunk_size)):

        data = input_data[i:i+chunk_size]
        batch_labels, batch_strs, batch_tokens = batch_converter(data)

        # Extract embeddings (on CPU)
        with torch.no_grad():
            results = model(batch_tokens, repr_layers=[12])

        emb = results["representations"][12].numpy()

        for idx, x in enumerate(batch_strs): 
            embeddings[i+idx:i+chunk_size+idx, :] = np.mean(emb[idx, 1:1+len(x),:], axis=0)

    print(embeddings.shape)

    return embeddings

# To generate RNA-FM embeddings
# mirna_emb = Featurizer(mirna_data, chunk_size = 20)
# sirna_emb = Featurizer(sirna_data, chunk_size = 20)
# pirna_emb = Featurizer(pirna_data, chunk_size = 20)

In [ ]:
# To load previously computed embeddings
# # Data-Whole
# mirna_emb = np.load('./feat_emb_correlation/unique/whole/mirna_embeddings.npy')
# sirna_emb = np.load('./feat_emb_correlation/unique/whole/sirna_embeddings.npy')
# pirna_emb = np.load('./feat_emb_correlation/unique/whole/pirna_embeddings.npy')

# Data-23K
mirna_emb = np.load('./feat_emb_correlation/unique/23k/mirna_embeddings_23k.npy')
sirna_emb = np.load('./feat_emb_correlation/unique/23k/sirna_embeddings_23k.npy')
pirna_emb = np.load('./feat_emb_correlation/unique/whole/pirna_embeddings.npy')

# Data-3K 
# mirna_emb = np.load('./feat_emb_correlation/unique/3k/mirna_embeddings_3k.npy')
# sirna_emb = np.load('./feat_emb_correlation/unique/3k/sirna_embeddings_3k.npy')
# pirna_emb = np.load('./feat_emb_correlation/unique/3k/pirna_embeddings_3k.npy')

In [ ]:
def generateIndividualFeaturePlots(all_embeddings, siRNA, miRNA, piRNA, output_file_path, dataNum, individual_format="svg", rasterize_points=True, raster_dpi=180):
    plt.rcParams.update({
        "svg.fonttype": "none",
        "font.family": "Arial",
        "font.size": 9,
        "axes.titlesize": 11,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "legend.title_fontsize": 9
    })

    reNameDict = {
        'gibbs_free_energy': 'Gibbs Free Energy',
        'minimum_free_energy': 'Minimum Free Energy',
        'melting_temp': 'Melting Temperature',
        'shannon_entropy': 'Shannon Entropy',
        'gc_content': 'GC Content',
        'au_skew': 'AU Skew',
        'gc_skew': 'GC Skew',
        'a_content': 'Nucleotide A',
        'u_content': 'Nucleotide U',
        'g_content': 'Nucleotide G',
        'c_content': 'Nucleotide C',
        'U_1': 'U at Position 1',
        'G_1': 'G at Position 1'
    }

    tsne = TSNE(n_components=2, random_state=42, perplexity=30, learning_rate=200)
    reduced_data = tsne.fit_transform(all_embeddings)
    siRNA_df = siRNA
    miRNA_df = miRNA
    piRNA_df = piRNA

    siRNA_df['RNA Type'] = 'siRNA'
    miRNA_df['RNA Type'] = 'miRNA'
    piRNA_df['RNA Type'] = 'piRNA'

    full_df = pd.concat([siRNA_df, miRNA_df, piRNA_df], ignore_index=True)
    RNA_types = full_df['RNA Type'].values
    type_colors = {'siRNA': '#1f77b4', 'miRNA': '#ff7f0e', 'piRNA': '#2ca02c'}
    color_vector = [type_colors[rna] for rna in RNA_types]

    common_cols = set(siRNA_df.columns) & set(miRNA_df.columns) & set(piRNA_df.columns)
    valid_features = [f for f in reNameDict if f in common_cols ]
    print(valid_features)

    os.makedirs(output_file_path, exist_ok=True)

    def get_feature_values(feature):
        vals = np.concatenate([
            siRNA_df[feature].values,
            miRNA_df[feature].values,
            piRNA_df[feature].values
        ])
        return vals

    def rscatter(ax, *args, **kwargs):
        # ensure only the heavy point cloud is rasterized
        if rasterize_points and "rasterized" not in kwargs:
            kwargs["rasterized"] = True
        return ax.scatter(*args, **kwargs)

    ncols = 4
    total_plots = 4 + len(valid_features)
    nrows = int(math.ceil(total_plots / ncols))

    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols,
        figsize=(ncols*4.0, nrows*3.6),
        constrained_layout=True
    )
    if nrows == 1:
        axes = np.array([axes])
    axes = axes.reshape(nrows, ncols)

    for ax in axes.flat:
        ax.set_rasterization_zorder(0)

    mask_si = (RNA_types == 'siRNA')
    mask_mi = (RNA_types == 'miRNA')
    mask_pi = (RNA_types == 'piRNA')

    ax = axes[0,0]
    rscatter(ax, reduced_data[:,0], reduced_data[:,1], c="#d3d3d3", s=4, alpha=0.25)
    rscatter(ax, reduced_data[mask_si,0], reduced_data[mask_si,1], c=type_colors['siRNA'], s=6, alpha=0.9)
    ax.set_title("siRNA only"); ax.set_xlabel("Dimension 1"); ax.set_ylabel("Dimension 2"); ax.grid(True, alpha=0.3)

    ax = axes[0,1]
    rscatter(ax, reduced_data[:,0], reduced_data[:,1], c="#d3d3d3", s=4, alpha=0.25)
    rscatter(ax, reduced_data[mask_mi,0], reduced_data[mask_mi,1], c=type_colors['miRNA'], s=6, alpha=0.9)
    ax.set_title("miRNA only"); ax.set_xlabel("Dimension 1"); ax.set_ylabel("Dimension 2"); ax.grid(True, alpha=0.3)

    ax = axes[0,2]
    rscatter(ax, reduced_data[:,0], reduced_data[:,1], c="#d3d3d3", s=4, alpha=0.25)
    rscatter(ax, reduced_data[mask_pi,0], reduced_data[mask_pi,1], c=type_colors['piRNA'], s=6, alpha=0.9)
    ax.set_title("piRNA only"); ax.set_xlabel("Dimension 1"); ax.set_ylabel("Dimension 2"); ax.grid(True, alpha=0.3)

    ax = axes[0,3]
    rscatter(ax, reduced_data[:, 0], reduced_data[:, 1], c=color_vector, s=5, alpha=0.7)
    for label in type_colors:
        ax.scatter([], [], c=type_colors[label], label=label)
    ax.legend(title="RNA Type", frameon=False, loc='lower right')
    ax.set_title("All RNA types")
    ax.set_xlabel("Dimension 1"); ax.set_ylabel("Dimension 2"); ax.grid(True, alpha=0.3)

    start_idx = 4
    for idx, feature in enumerate(valid_features):
        r = (start_idx + idx) // ncols
        c = (start_idx + idx) % ncols
        ax = axes[r, c]
        vals = get_feature_values(feature)
        sc = rscatter(ax, reduced_data[:,0], reduced_data[:,1], c=vals, cmap='coolwarm', s=5, alpha=0.8)
        ax.set_title(reNameDict[feature])
        ax.set_xlabel("Dimension 1"); ax.set_ylabel("Dimension 2"); ax.grid(True, alpha=0.3)
        cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=7)

    last_filled = total_plots - 1
    for k in range(last_filled + 1, nrows*ncols):
        r = k // ncols
        c = k % ncols
        axes[r, c].axis('off')

    panel_path = os.path.join(output_file_path, f"RNA_FM_tSNE_PANEL_{dataNum}k.{individual_format}")
    # fig.suptitle(f"t-SNE for RNA-FM embeddings — {dataNum}k", fontsize=12, y=1.02)
    fig.savefig(panel_path, format=individual_format, bbox_inches="tight", dpi=raster_dpi)
    plt.close(fig)
    print(f"Saved {individual_format.upper()} panel to: {panel_path}")


all_emb = np.concat([sirna_emb, mirna_emb, pirna_emb])
print(all_emb.shape)
generateIndividualFeaturePlots(all_emb, sirna, mirna, pirna, "./", "23")

## 4.2 Correlation 

### 4.2.1 Feature Initialization

In [ ]:
gc_content = list(mirna['gc_content'].values) + list(sirna['gc_content'].values) + list(pirna['gc_content'].values)
gfe = list(mirna['gibbs_free_energy'].values) + list(sirna['gibbs_free_energy'].values) + list(pirna['gibbs_free_energy'].values)
mfe = list(mirna['minimum_free_energy'].values) + list(sirna['minimum_free_energy'].values) + list(pirna['minimum_free_energy'].values)
a_content = list(mirna['a_content'].values) + list(sirna['a_content'].values) + list(pirna['a_content'].values)
u_content = list(mirna['u_content'].values) + list(sirna['u_content'].values) + list(pirna['u_content'].values)
g_content = list(mirna['g_content'].values) + list(sirna['g_content'].values) + list(pirna['g_content'].values)
c_content = list(mirna['c_content'].values) + list(sirna['c_content'].values) + list(pirna['c_content'].values)
tm = list(mirna['melting_temp'].values) + list(sirna['melting_temp'].values) + list(pirna['melting_temp'].values)
au_skew = list(mirna['au_skew'].values) + list(sirna['au_skew'].values) + list(pirna['au_skew'].values)
gc_skew = list(mirna['gc_skew'].values) + list(sirna['gc_skew'].values) + list(pirna['gc_skew'].values)
shannon = list(mirna['shannon_entropy'].values) + list(sirna['shannon_entropy'].values) + list(pirna['shannon_entropy'].values)
num_palindromes = list(mirna['num_palindromes'].values) + list(sirna['num_palindromes'].values) + list(pirna['num_palindromes'].values)
u1 = list(mirna['U_1'].values) + list(sirna['U_1'].values) + list(pirna['U_1'].values)
g1 = list(mirna['G_1'].values) + list(sirna['G_1'].values) + list(pirna['G_1'].values)
ggall = list(mirna['GG_all'].values) + list(sirna['GG_all'].values) + list(pirna['GG_all'].values)
uaall = list(mirna['UA_all'].values) + list(sirna['UA_all'].values) + list(pirna['UA_all'].values)
ccall = list(mirna['CC_all'].values) + list(sirna['CC_all'].values) + list(pirna['CC_all'].values)
gcall = list(mirna['GC_all'].values) + list(sirna['GC_all'].values) + list(pirna['GC_all'].values)
uuall = list(mirna['UU_all'].values) + list(sirna['UU_all'].values) + list(pirna['UU_all'].values)

all_emb = np.concat([mirna_emb, sirna_emb, pirna_emb])
print(all_emb.shape)

In [ ]:
def gen_corr_feat(embeddings, feature, type, filename=None, corr_list=False):
    """
    embeddings: types of embeddings eg. [mirna_emb, sirna_emb, pirna_emb]
    feature: list of the features eg. gc_content
    type: Pearson or Spearman
    filename: saved filename
    corr_list: to generate output as list 
    """

    corr = []
    pval = []
    labels = ['miRNA', 'siRNA', 'piRNA']

    # DataFrame
    df = pd.DataFrame()
    CorrType = []
    RNAType = []
    RnafmEmb = []
    Correlation = []
    PVal = []

    pos = 0
    if type == 'Pearson':
        for idx, emb in enumerate(embeddings):
            print(idx, emb.shape)
            for i in range(0, 640):
                corr.append(pearsonr(feature[pos:pos+len(emb)],emb[:,i])[0])
                pval.append(pearsonr(feature[pos:pos+len(emb)],emb[:,i])[1])
            pos += len(emb)

    elif type == 'Spearman':
        for idx, emb in enumerate(embeddings):
            print(idx, emb.shape)
            for i in range(0, 640):
                corr.append(spearmanr(feature[pos:pos+len(emb)],emb[:,i])[0])
                pval.append(spearmanr(feature[pos:pos+len(emb)],emb[:,i])[1])
            pos += len(emb)

    print(len(corr), len(pval))

    #DataFrame
    for i, x in enumerate([corr[:640],corr[640:640*2],corr[640*2:]]):
        for y in np.argsort(x)[::-1].tolist()[:20]:
            CorrType.append('positive')
            RNAType.append(labels[i])
            RnafmEmb.append(y)
            Correlation.append(x[y])
            PVal.append([pval[:640], pval[640:640*2], pval[640*2:]][i][y])

        for y in np.argsort(x).tolist()[:20]: 
            CorrType.append('negative')
            RNAType.append(labels[i])
            RnafmEmb.append(y)
            Correlation.append(x[y])
            PVal.append([pval[:640], pval[640:640*2], pval[640*2:]][i][y])

    df['RNA Type'] = RNAType
    df['Type'] = CorrType
    df['RNA-FM Features'] = RnafmEmb
    df['Correlation'] = Correlation
    df['P-Value'] = PVal

    if filename != None:
        df.to_csv(filename)

    return df, corr

### 4.2.2 Correlation Calculation

In [ ]:
gc_corr_df_23, gc_corr_23 = gen_corr_feat([mirna_emb, sirna_emb, pirna_emb], gc_content, 'Pearson')

gfe_corr_df_23, gfe_corr_23 = gen_corr_feat([mirna_emb, sirna_emb, pirna_emb], gfe,'Pearson')

mfe_corr_df_23, mfe_corr_23 = gen_corr_feat([mirna_emb, sirna_emb, pirna_emb], mfe,'Pearson')

tm_corr_df_23, tm_corr_23  = gen_corr_feat([mirna_emb, sirna_emb, pirna_emb], tm, 'Pearson')

shannon_corr_df_23, shannon_corr_23= gen_corr_feat([mirna_emb, sirna_emb, pirna_emb], shannon, 'Pearson')

a_corr_df_23, a_corr_23 = gen_corr_feat([mirna_emb, sirna_emb, pirna_emb], a_content,'Pearson')

u_corr_df_23, u_corr_23 = gen_corr_feat([mirna_emb, sirna_emb, pirna_emb], u_content,'Pearson')

g_corr_df_23, g_corr_23 = gen_corr_feat([mirna_emb, sirna_emb, pirna_emb], g_content,'Pearson')

c_corr_df_23, c_corr_23 = gen_corr_feat([mirna_emb, sirna_emb, pirna_emb], c_content,'Pearson')

au_skew_corr_df_23, au_skew_corr_23  = gen_corr_feat([mirna_emb, sirna_emb, pirna_emb], au_skew, 'Pearson')

gc_skew_corr_df_23, gc_skew_corr_23  = gen_corr_feat([mirna_emb, sirna_emb, pirna_emb], gc_skew, 'Pearson')

u1_corr_df_23, u1_corr_23 = gen_corr_feat([mirna_emb, sirna_emb, pirna_emb], u1,'Pearson')

g1_corr_df_23, g1_corr_23 = gen_corr_feat([mirna_emb, sirna_emb, pirna_emb], g1,'Pearson')

ggall_corr_df_23, ggall_corr_23 = gen_corr_feat([mirna_emb, sirna_emb, pirna_emb], ggall,'Pearson')

uaall_corr_df_23, uaall_corr_23 = gen_corr_feat([mirna_emb, sirna_emb, pirna_emb], uaall,'Pearson')

ccall_corr_df_23, ccall_corr_23 = gen_corr_feat([mirna_emb, sirna_emb, pirna_emb], ccall,'Pearson')

gcall_corr_df_23, gcall_corr_23 = gen_corr_feat([mirna_emb, sirna_emb, pirna_emb], gcall,'Pearson')

uuall_corr_df_23, uuall_corr_23 = gen_corr_feat([mirna_emb, sirna_emb, pirna_emb], uuall,'Pearson')

In [ ]:
labels = ['GC Content', 'Gibbs Free Energy', 'Minimum Free Energy', 'Nucleotide A', 'Nucleotide U', 'Nucleotide G', 'Nucleotide C', 'Melting Temperature', 'AU Skew', 'GC Skew', 'Shannon Entropy', 'U at Position 1', 'G at Position 1', 'GG across Sequence', 'UA across Sequence', 'CC across Sequence', 'GC across Sequence', 'UU across Sequence']
features = []
for y in labels:
    features.append([y]*len(gc_corr_df_23))
features = [feat for featlist in features for feat in featlist]
feat_df_23 = pd.concat([gc_corr_df_23, gfe_corr_df_23, mfe_corr_df_23, a_corr_df_23, u_corr_df_23, g_corr_df_23, c_corr_df_23, tm_corr_df_23, au_skew_corr_df_23, gc_skew_corr_df_23, shannon_corr_df_23, u1_corr_df_23, g1_corr_df_23, ggall_corr_df_23, uaall_corr_df_23, ccall_corr_df_23, gcall_corr_df_23, uuall_corr_df_23])
feat_df_23['Interpretable Features'] = features
feat_df_23 = feat_df_23[[feat_df_23.columns.tolist()[-1]] + feat_df_23.columns.tolist()[:-1]]
feat_df_23

In [ ]:
# All Types Combined
all_emb_gc_corr_23 = []
all_emb_gfe_corr_23 = []
all_emb_mfe_corr_23 = []
all_emb_a_corr_23 = []
all_emb_u_corr_23 = []
all_emb_g_corr_23 = []
all_emb_c_corr_23 = []
all_emb_tm_corr_23 = [] #melting T
all_emb_auskew_corr_23 = [] #au_skew
all_emb_gcskew_corr_23 = [] #gc_skew
all_emb_shannon_corr_23 = [] #shannon E

for idx in range(0, 640):
    all_emb_gc_corr_23.append(pearsonr(gc_content, all_emb[:,idx])[0]) 
    all_emb_gfe_corr_23.append(pearsonr(gfe, all_emb[:,idx])[0]) 
    all_emb_mfe_corr_23.append(pearsonr(mfe, all_emb[:,idx])[0]) 
    all_emb_a_corr_23.append(pearsonr(a_content, all_emb[:,idx])[0]) 
    all_emb_u_corr_23.append(pearsonr(u_content, all_emb[:,idx])[0]) 
    all_emb_g_corr_23.append(pearsonr(g_content, all_emb[:,idx])[0]) 
    all_emb_c_corr_23.append(pearsonr(c_content, all_emb[:,idx])[0]) 
    all_emb_tm_corr_23.append(pearsonr(tm, all_emb[:,idx])[0]) 
    all_emb_auskew_corr_23.append(pearsonr(au_skew, all_emb[:,idx])[0]) 
    all_emb_gcskew_corr_23.append(pearsonr(gc_skew, all_emb[:,idx])[0]) 
    all_emb_shannon_corr_23.append(pearsonr(shannon, all_emb[:,idx])[0]) 

all_emb_td_dict_23 = {'u1': [], 'g1': [], 'ggall': [], 'uaall': [], 'ccall': [], 'gcall': [], 'uuall': []}

for idx in range(0, 640):
    all_emb_td_dict_23['u1'].append(pearsonr(u1, all_emb[:,idx])[0])
    all_emb_td_dict_23['g1'].append(pearsonr(g1, all_emb[:,idx])[0])

    all_emb_td_dict_23['ggall'].append(pearsonr(ggall, all_emb[:,idx])[0])
    all_emb_td_dict_23['uaall'].append(pearsonr(uaall, all_emb[:,idx])[0])
    all_emb_td_dict_23['ccall'].append(pearsonr(ccall, all_emb[:,idx])[0])
    all_emb_td_dict_23['gcall'].append(pearsonr(gcall, all_emb[:,idx])[0])
    all_emb_td_dict_23['uuall'].append(pearsonr(uuall, all_emb[:,idx])[0])

In [ ]:
all_df_23 = pd.DataFrame()
all_features = [] 
RnafmEmb = []
PCorrelation = [] 
Type =[]
PPVal = []

labels = ['GC Content', 'Gibbs Free Energy', 'Minimum Free Energy', 'Nucleotide A','Nucleotide U','Nucleotide G','Nucleotide C', 'Melting Temperature', 'AU Skew', 'GC Skew', 'Shannon Entropy', 'U at Position 1','G at Position 1',  'GG across Sequence', 'UA across Sequence', 'CC across Sequence', 'GC across Sequence', 'UU across Sequence']
pval_ref = [gc_content, gfe, mfe, a_content, u_content, g_content, c_content, tm, au_skew, gc_skew, shannon, u1, g1, ggall, uaall, ccall, gcall, uuall]

for i, x in enumerate([all_emb_gc_corr_23, all_emb_gfe_corr_23, all_emb_mfe_corr_23, all_emb_a_corr_23, all_emb_u_corr_23, all_emb_g_corr_23, all_emb_c_corr_23, all_emb_tm_corr_23, all_emb_auskew_corr_23, all_emb_gcskew_corr_23,all_emb_shannon_corr_23] + list(all_emb_td_dict_23.values())):
    for y in np.argsort(x)[::-1].tolist()[:20]:
        Type.append('positive')
        all_features.append(labels[i])
        RnafmEmb.append(y)
        PCorrelation.append(x[y])
        PPVal.append(pearsonr(pval_ref[i], all_emb[:,y])[1])
    for y in np.argsort(x).tolist()[:20]:
        Type.append('negative')
        all_features.append(labels[i])
        RnafmEmb.append(y)
        PCorrelation.append(x[y])
        PPVal.append(pearsonr(pval_ref[i], all_emb[:,y])[1])

all_df_23['Interpretable Features'] = all_features
all_df_23['RNA Type'] = ['AllTypes'] * len(all_features)
all_df_23['Type'] = Type        
all_df_23['RNA-FM Features'] = RnafmEmb
all_df_23['Correlation'] = PCorrelation
all_df_23['P-Value'] = PPVal

# all_df_23.to_csv('pearsoncorr_misipirna_all.csv')
all_df_23

### 4.2.3 Correlation Heatmap

In [ ]:
# Top 20 Most Correlated RNA-FM Features for Base Features Compiled 
def generate_base_feat(feat_corr, all_emb_feat_corr, labels = ['GC Content','Gibbs Free Energy', 'Minimum Free Energy', 'Nucleotide A', 'Nucleotide U','Nucleotide G','Nucleotide C','Melting Temperature','AU Skew','GC Skew','Shannon Entropy', 'U at Position 1', 'G at Position 1', 'GG across Sequence', 'UA across Sequence', 'CC across Sequence', 'GC across Sequence', 'UU across Sequence']):
    """
    feat_corr: list containing feature values eg. gc_content
    all_emb_feat_corr: list containing feature values of the Irrespective of RNA type/All Types eg. all_emb_gc_corr
    labels: labels for each feature 
    """
    
    rna_type_labels = ['miRNA', 'siRNA','piRNA','AllTypes']

    indices = []
    for i, feature in enumerate(feat_corr):
        # Positive
        indices.append(np.argsort(feature[:640])[::-1][:20].tolist()) #miRNA 
        indices.append(np.argsort(feature[640:640*2])[::-1][:20].tolist()) #siRNA 
        indices.append(np.argsort(feature[640*2:])[::-1][:20].tolist()) #piRNA
        # Negative
        indices.append(np.argsort(feature[:640])[:20].tolist()) #miRNA
        indices.append(np.argsort(feature[640:640*2])[:20].tolist()) #siRNA 
        indices.append(np.argsort(feature[640*2:])[:20].tolist()) #piRNA

    rows = []

    df = pd.DataFrame(columns=['Base Feature'] + list(set([b for a in indices for b in a])))
    for i, feature in enumerate(feat_corr):
        row1 = []
        row2 = []
        row3 = []
        row4 = []
        for j in list(set([b for a in indices for b in a])):
            row1.append(feature[0+j])
            row2.append(feature[640+j])
            row3.append(feature[640*2+j])
            row4.append(all_emb_feat_corr[i][0+j])
        for id, z in enumerate([row1, row2, row3, row4]):
            z.insert(0, rna_type_labels[id]+' '+labels[i])
            rows.append(z)

    for y in range(len(rows)):
        df.loc[y,:] = rows[y]

    df[df.columns[1:]] = df[df.columns[1:]].astype(float)

    return df 

In [ ]:
# 23K
comb_df_23 = generate_base_feat([gc_corr_23, gfe_corr_23, mfe_corr_23, a_corr_23, u_corr_23, g_corr_23, c_corr_23, tm_corr_23, au_skew_corr_23, gc_skew_corr_23, shannon_corr_23, u1_corr_23, g1_corr_23, ggall_corr_23, uaall_corr_23, ccall_corr_23, gcall_corr_23, uuall_corr_23],
                                [all_emb_gc_corr_23, all_emb_gfe_corr_23, all_emb_mfe_corr_23, all_emb_a_corr_23, all_emb_u_corr_23, all_emb_g_corr_23, all_emb_c_corr_23, all_emb_tm_corr_23, all_emb_auskew_corr_23, all_emb_gcskew_corr_23, all_emb_shannon_corr_23, [x for x in all_emb_td_dict_23['u1']], [x for x in all_emb_td_dict_23['g1']], [x for x in all_emb_td_dict_23['ggall']], [x for x in all_emb_td_dict_23['uaall']], [x for x in all_emb_td_dict_23['ccall']], [x for x in all_emb_td_dict_23['gcall']], [x for x in all_emb_td_dict_23['uuall']]])

In [ ]:
network_df = comb_df_23.set_index(['Base Feature'])
fig, ax = plt.subplots(figsize=(20, 20))

g = nx.MultiGraph()
# instead of threshold, pick top 2
network_dict = {}
for y in network_df.T:
    network_dict[y] = sorted(network_df.T[y].items(), key=lambda x: x[1])[:2] + sorted(network_df.T[y].items(), key=lambda x: x[1], reverse=True)[:2]
    if y.startswith('AllTypes'):
        g.add_node('\n'.join(y.split(' ')[1:]))
        for rna_type in ['AllTypes', 'siRNA', 'miRNA', 'piRNA']:
            for z in [x for x in network_dict[y.replace('AllTypes', rna_type)]]:
                if z[0] != y: 
                    g.add_node(z[0])
                    g.add_edge('\n'.join(y.split(' ')[1:]), z[0], weight=z[1], label=rna_type)
                    # print(y, z[0], z[1])
print(g)

hex_colors = ['#B85F69', '#CB9098', '#9E577A'] 

colormap = []
for node,_ in g.nodes.items():
    if type(node) is int:
        colormap.append(matplotlib.colors.to_hex('tab:brown')) 
    elif node in ['Nucleotide\nA', 'Nucleotide\nU', 'Nucleotide\nG', 'Nucleotide\nC', 'GC\nContent', 'GC\nSkew', 'AU\nSkew', 'Shannon\nEntropy']: #Structural 
        colormap.append(hex_colors[0]) #[4])
    elif node in ['U\nat\nPosition\n1', 'UU\nacross\nSequence','G\nat\nPosition\n1', 'GG\nacross\nSequence', 'CC\nacross\nSequence', 'GC\nacross\nSequence', 'UA\nacross\nSequence']: # Positional 
        colormap.append(hex_colors[1])#[0])
    elif node in ['Gibbs\nFree\nEnergy', 'Minimum\nFree\nEnergy', 'Melting\nTemperature']: #Thermo
        colormap.append(hex_colors[2])

edges, weights = zip(*nx.get_edge_attributes(g,'weight').items())
nx.set_edge_attributes(g,values={k: abs(v) for k, v in nx.get_edge_attributes(g,'weight').items()},name='weight') # such that weights are positive for kamada_kawai, but colours will represent neg/pos correlation
paths = dict(nx.shortest_path_length(g, weight=weights))
for x in paths:
    for y in paths[x]:
        paths[x][y] = paths[x][y] * 0.75
d = dict(g.degree)

df = pd.DataFrame(index=g.nodes(), columns=g.nodes())
for row, data in nx.shortest_path_length(g):
    for col, dist in data.items():
        df.loc[row,col] = dist

df = df.fillna(df.max().max())

pos = nx.kamada_kawai_layout(g, dist=df.to_dict(), scale=1.5)
shapes = [] 
nodesize = []
for y in g.nodes():
    if type(y) is not int:
        shapes.append('o')
        nodesize.append(5500)
    else:
        shapes.append('p')
        nodesize.append(1500) #*(8-2)*(d[90]-1)/(21-1))
rna_dict = {'AllTypes': 'tab:purple',
            'siRNA':'tab:blue',
            'miRNA':'tab:orange',
            'piRNA':'tab:green'}

edge_style = []
for weight in weights:
    if weight < 0: 
        edge_style.append('dotted')
    elif weight >= 0:
        edge_style.append('solid')

edge_width = [x*8 for x in list(map(abs,weights))] #13

options = {'edgecolors':'black', 
    'linewidths': 0.5}

for i, node in enumerate(g.nodes()):
    if node in [609,494]:
        pos[node] = pos[node] + [0.15, -0.01]
    elif node in [50, 'Nucleotide\nG']: 
        pos[node] = pos[node] + [0.03, -0.03]
    elif node in [402]: 
        pos[node] = pos[node] + [-0.1, 0.5]
    elif node in [131]:
        pos[node] = pos[node] + [-0.1, -0.1]
    elif node in [148]:
        pos[node] = pos[node] + [-0.2, -0.15]
    elif node in ['GC\nContent', 'Nucleotide\nU']:
        pos[node] = pos[node] + [-0.05, -0.05]
    elif node in ['UA\nacross\nSequence',]:
        pos[node] = pos[node] + [0.2, 0.04]
    elif node in ['Gibbs\nFree\nEnergy']:
        pos[node] = pos[node] + [0.12, -0.06]
    elif node in [204]:
        pos[node] = pos[node] + [-0.08, 0.07]
    elif node in ['Minimum\nFree\nEnergy']:
        pos[node] = pos[node] + [-0.01, -0.015]
    elif node in [512]:
        pos[node] = pos[node] + [-0.015, 0.03]
    elif node in [481]:
        pos[node] = pos[node] + [0.01, 0.03]
    elif node in [556]:
        pos[node] = pos[node] + [0.02, 0.05]
    elif node in [603]:
        pos[node] = pos[node] + [0.05, -0.03]
    elif node in [318, 82, 561, 'GC\nacross\nSequence', 121, 208]:
        pos[node] = pos[node] + [0.2, -0.1]
    elif node in [374]:
        pos[node] = pos[node] + [0, -0.2]
    elif node in [90]:
        pos[node] = pos[node] + [0.08, -0.14]
    elif node in ['U\nat\nPosition\n1']:
        pos[node] = pos[node] + [0.35, 0.3]
    elif node in [40, 539, 628, 410, 250, 490, 408, 411]:
        pos[node] = pos[node] + [0.35, 0.4]
    elif node in [86, 451, 128, 156, 476]:
        pos[node] = pos[node] + [0.03, 0.1]
    elif node in [116]:
        pos[node] = pos[node] + [-0.03, 0]

edge_counter = {}
for edge in g.edges():
    if edge not in edge_counter.keys():
        edge_counter[edge] = 1
    else:
        edge_counter[edge] += 1 

edge_colour = []
types = ['arc3', 'arc3,rad=0.2', 'arc3,rad=0.4', 'arc3,rad=-0.4']
connections = []
for edge in nx.get_edge_attributes(g, 'label'):
    edge_colour.append(rna_dict[nx.get_edge_attributes(g, 'label')[edge]])
for unique_edge in edge_counter:
    if edge_counter[unique_edge ] == 4:  
        connections.append('arc3')
        connections.append('arc3, rad=0.3')
        connections.append('arc3, rad=0.5')
        connections.append('arc3, rad=-0.5')
    elif edge_counter[unique_edge ] == 3:
        connections.append('arc3')
        connections.append('arc3, rad=0.5')
        connections.append('arc3, rad=-0.5')
    elif edge_counter[unique_edge ] == 2:
        connections.append('arc3')#, rad=0.5')
        connections.append('arc3, rad=-0.2')
    elif edge_counter[unique_edge ] == 1:
        connections.append('arc3')

for count, node in enumerate(g.nodes()):
    if type(node) == int: 
        nx.draw_networkx_nodes(g, pos, nodelist = [node], node_color='tab:brown', node_shape = shapes[count], node_size = nodesize[count], **options) 
        nx.draw_networkx_labels(g, pos, {node: node}, font_size=12, font_color='white')
    else: 
        nx.draw_networkx_nodes(g, pos, nodelist = [node], node_color=colormap[count], node_shape = shapes[count], node_size = nodesize[count], **options)
        nx.draw_networkx_labels(g, pos, {node: node}, font_size=12, font_color='white')

for id, edge in enumerate(g.edges()):
    nx.draw_networkx_edges(g, pos, edgelist=[edge], arrows=True, edge_color=edge_colour[id],width=4, style=edge_style[id], connectionstyle=connections[id], alpha=[(abs(x) - min(map(abs, weights)))/(max(map(abs, weights)) - min(map(abs, weights))) for x in weights][id])

legend_dict = {'RNA-FM Features': 'tab:brown', 
             'Structural Features': hex_colors[0],
             'Positional Features': hex_colors[1],
             'Thermodynamic Features':hex_colors[2]
             }
handles = [Patch(facecolor=legend_dict [label], label=label) for label in legend_dict ]
ax.add_artist(ax.legend(handles=handles, loc=(0.05,0.85), title='Nodes', fontsize=14, title_fontsize=16))
ax.axis('off')

legend2_dict = {'siRNA': 'tab:blue',
                'miRNA': 'tab:orange',
                'piRNA': 'tab:green',
                'AllTypes': 'tab:purple'
                }
handles2 = [Patch(facecolor=legend2_dict[label], label=label) for label in legend2_dict]
handles2.append(matplotlib.lines.Line2D([], [], label='Positive Correlation', color='black'))
handles2.append(matplotlib.lines.Line2D([], [], label='Negative Correlation', color='black', linestyle=':'))
handles2.append(matplotlib.lines.Line2D([], [], marker='o', markersize=6, label='|Low Correlation| between Nodes', color=(0, 0, 0, 0.2), markerfacecolor='black', markeredgecolor='black'))
handles2.append(matplotlib.lines.Line2D([], [], marker='o', markersize=6, label='|High Correlation| between Nodes',color='black'))
ax.add_artist(ax.legend(handles=handles2, loc=(0.05,0.74), title='Edges', fontsize=14, title_fontsize=16))

colors1 = plt.cm.Greys(np.linspace(1, 0, 100))
colors2 = plt.cm.Greys(np.linspace(0, 1, 100))
colors = np.vstack((colors1, colors2))
sm = plt.cm.ScalarMappable(cmap=matplotlib.colors.LinearSegmentedColormap.from_list('my_colormap', colors), norm=plt.Normalize(vmin = -1, vmax=1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.3)
cbar.ax.tick_params(labelsize=14)
cbar.ax.set_title('Edge Opacity', size=16, y=1.02)
cbar.set_label("Pearson Correlation Coefficient", fontsize=14)
plt.tight_layout()
# plt.savefig(f"./pcc_23k_rnafm_network_v3_alpha_2_final_shannon_v1.png", dpi=300)
# plt.savefig(f"./pcc_23k_rnafm_network_v3_alpha_2_final_shannon_v1.svg", dpi=600)
# plt.show()

In [ ]:
df_to_use = comb_df_23
data = df_to_use[~df_to_use['Base Feature'].str.startswith('AllTypes')]

pal = sns.color_palette('deep',20)
hex_colors = list(pal.as_hex())
hex_colors = [x for x in hex_colors if x != '#dd8452']
hex_colors = [x for x in hex_colors if x != '#8c8c8c']
print(hex_colors)
hierarchy.set_link_color_palette(hex_colors)

# Plot
g = sns.clustermap(data[data.columns[1:]].T, cmap="coolwarm", xticklabels=[(' ').join(x[0:]) for x in data['Base Feature'].str.split(' ')], figsize=(16,20), dendrogram_ratio=0.1, yticklabels=False, metric='euclidean', method='ward')

g.ax_cbar.set_ylim((-1,1))

x_labels = [item.get_text() for item in g.ax_heatmap.get_xticklabels()]

for i in range(0, len(g.ax_heatmap.get_xticklabels())):
    if g.ax_heatmap.get_xticklabels()[i].get_text().startswith('siRNA'):
        g.ax_heatmap.get_xticklabels()[i].set_color('tab:blue')
        g.ax_heatmap.get_xticklabels()[i].set_fontsize(20)
        x_labels[i] = (' ').join(g.ax_heatmap.get_xticklabels()[i].get_text().split(' ')[1:])
    elif g.ax_heatmap.get_xticklabels()[i].get_text().startswith('miRNA'):
        g.ax_heatmap.get_xticklabels()[i].set_color('tab:orange')
        g.ax_heatmap.get_xticklabels()[i].set_fontsize(20)
        x_labels[i] = (' ').join(g.ax_heatmap.get_xticklabels()[i].get_text().split(' ')[1:])
    elif g.ax_heatmap.get_xticklabels()[i].get_text().startswith('piRNA'):
        g.ax_heatmap.get_xticklabels()[i].set_color('tab:green')
        g.ax_heatmap.get_xticklabels()[i].set_fontsize(20)
        x_labels[i] = (' ').join(g.ax_heatmap.get_xticklabels()[i].get_text().split(' ')[1:])

g.ax_heatmap.set_xticklabels(x_labels)

plt.tight_layout()

lut2 = dict(zip(['siRNA', 'miRNA', 'piRNA'], ['tab:blue','tab:orange','tab:green']))
lut2 = OrderedDict(lut2)
lut2 = {k : lut2[k] for k in ['siRNA', 'miRNA', 'piRNA']}

hm_pos = g.ax_heatmap.get_position()
print(hm_pos)

handles2 = [Patch(facecolor=lut2[name]) for name in lut2]
legend2 = plt.legend(handles2, lut2, ncol=1, loc=(22,0.55))
plt.gca().add_artist(legend2)

row_den = hierarchy.dendrogram(g.dendrogram_row.linkage,distance_sort=True, above_threshold_color= 'gray', color_threshold=3.9,ax=g.ax_row_dendrogram, orientation='left')

col_den = hierarchy.dendrogram(g.dendrogram_col.linkage,distance_sort=True, above_threshold_color= 'gray', color_threshold=5,ax=g.ax_col_dendrogram, orientation='top')

rd_pos = g.ax_row_dendrogram.get_position()
cd_pos = g.ax_col_dendrogram.get_position()
g.ax_row_dendrogram.set_position([rd_pos.x0+0.015, rd_pos.y0, rd_pos.width, rd_pos.height]) # -0.015, rd_pos.y0, rd_pos.width, rd_pos.height]) 
g.ax_col_dendrogram.set_position([cd_pos.x0, cd_pos.y0*0.9, cd_pos.width, cd_pos.height*0.86])  # [x, y, width, height]

cmap_pos = g.ax_cbar.get_position()
g.ax_cbar.set_position((rd_pos.x0+0.01, 0.82, 0.04, 0.08))
g.ax_cbar.set_ylabel('Pearson Correlation Coefficient', loc='center',labelpad=-80)
print(g.ax_cbar.get_position())
for spine in g.ax_cbar.spines.values():
    spine.set_visible(True)

# plt.savefig('clustermap_scc_alloligo_3k_top20_individual_comb_pub_wopal_final.png',dpi=300)
# plt.savefig('clustermap_scc_alloligo_3k_top20_individual_comb_pub_wopal_final_large.svg',dpi=600)
plt.show()

In [ ]:
# Irrespective only
data = comb_df_23[comb_df_23['Base Feature'].str.startswith('AllTypes')]
data = data.replace({'AllTypes u @ pos1': 'AllTypes U at Position 1','AllTypes g @ pos1': 'AllTypes G at Position 1','AllTypes gg @ all': 'AllTypes GG across Sequence','AllTypes ua @ all': 'AllTypes UA across Sequence','AllTypes cc @ all': 'AllTypes CC across Sequence','AllTypes gc @ all': 'AllTypes GC across Sequence','AllTypes uu @ all': 'AllTypes UU across Sequence'})

pal = sns.color_palette('deep',20)
hex_colors = list(pal.as_hex())
hex_colors = [x for x in hex_colors if x != '#dd8452']
hex_colors = [x for x in hex_colors if x != '#8c8c8c']
print(hex_colors)
hierarchy.set_link_color_palette(hex_colors)

# Plot
g = sns.clustermap(data[data.columns[1:]].T, cmap="coolwarm", metric='euclidean', method='ward', xticklabels=[(' ').join(x[1:]) for x in data['Base Feature'].str.split(' ')], yticklabels=False,figsize=(12,15), dendrogram_ratio=0.1,cbar_pos=(0.06, 0.91, 0.03, 0.08))

plt.tight_layout()

hm_pos = g.ax_heatmap.get_position()

row_den = hierarchy.dendrogram(g.dendrogram_row.linkage,distance_sort=True, above_threshold_color= 'gray', color_threshold=2.75,ax=g.ax_row_dendrogram, orientation='left')
col_den = hierarchy.dendrogram(g.dendrogram_col.linkage,distance_sort=True, above_threshold_color= 'gray', color_threshold=5.75,ax=g.ax_col_dendrogram, orientation='top')

rd_pos = g.ax_row_dendrogram.get_position()
cd_pos = g.ax_col_dendrogram.get_position()

cmap_pos = g.ax_cbar.get_position()
g.ax_cbar.set_position((rd_pos.x0+0.02, 0.90, 0.04, 0.08))
g.ax_cbar.set_ylabel('Pearson Correlation Coefficient', loc='center',labelpad=-80)
print(g.ax_cbar.get_position())
g.ax_cbar.set_ylim((-1,1))
for spine in g.ax_cbar.spines.values():
    spine.set_visible(True)

# plt.savefig('clustermap_scc_alloligo_3k_top20_irrespective_comb_pub_wopal_final.png',dpi=300)
# plt.savefig('clustermap_scc_alloligo_3k_top20_irrespective_comb_pub_wopal_final.svg',dpi=600)
plt.show()

### 4.2.4 KDE Comparison 

In [ ]:
# Plot the distribution of the correlated values, RNA-type Specific
labels = ['GC Content', 'Gibbs Free Energy', 'Minimum Free Energy','Nucleotide A','Nucleotide U','Nucleotide G','Nucleotide C',  'Melting Temperature', 'AU Skew', 'GC Skew', 'Shannon Entropy', 'U at Position 1','G at Position 1',  'GG across Sequence', 'UA across Sequence', 'CC across Sequence', 'GC across Sequence', 'UU across Sequence']

temp_key = list(all_emb_td_dict_23.keys())
print(temp_key)

# PCC 
corr_23 = [gc_corr_23, gfe_corr_23, mfe_corr_23, a_corr_23, u_corr_23, g_corr_23, c_corr_23, tm_corr_23, au_skew_corr_23,gc_skew_corr_23, shannon_corr_23, u1_corr_23, g1_corr_23, ggall_corr_23, uaall_corr_23, ccall_corr_23, gcall_corr_23, uuall_corr_23] 

all_emb_corr_23 = [all_emb_gc_corr_23, all_emb_gfe_corr_23, all_emb_mfe_corr_23,all_emb_a_corr_23, all_emb_u_corr_23, all_emb_g_corr_23,all_emb_c_corr_23,all_emb_tm_corr_23, all_emb_auskew_corr_23, all_emb_gcskew_corr_23, all_emb_shannon_corr_23] + [[y for y in all_emb_td_dict_23[x]] for x in temp_key] 

fig, axs = plt.subplots(4, 5, figsize=(17.5,12.5))

axs[-1,-1].axis('off')
axs[-1,-2].axis('off')

yticks = []
for a in range(0, len(labels)):
    # Features used in Paper 
    if a in [1, 2]:
        yticks.append(np.arange(0, 4.1, 1))
    elif a in [0, 3, 4, 5, 7, 8, 6, 9, 15]:
        yticks.append(np.arange(0, 2.01, 0.5))
    elif a in [11]:
        yticks.append(np.arange(0, 8.1, 2))
    elif a in [12, 10]:
        yticks.append(np.arange(0, 6.1, 2))
    elif a in [13, 14, 16, 17]:
        yticks.append(np.arange(0, 3.1, 0.5))


for id, ax in enumerate(axs.flatten().tolist()[:len(labels)]):
    sns.kdeplot(corr_23[id][640:640*2], ax=ax, label="siRNA", color='tab:blue')
    sns.kdeplot(corr_23[id][:640], ax=ax, label="miRNA", color='tab:orange', linestyle=(0, (3, 1, 1, 1, 1, 1)))
    sns.kdeplot(corr_23[id][640*2:], ax=ax, label="piRNA", color='tab:green', linestyle='--')
    sns.kdeplot(all_emb_corr_23[id], ax=ax, label="AllTypes", color='tab:purple', linestyle='-.')
    ax.set_title(labels[id], fontsize=22)
    ax.set_xticks([-1, -0.5, -0.3, 0, 0.3, 0.5, 1], [-1, '-0.5    ', '  -0.3', 0, '0.3  ', '    0.5', 1], fontsize=17)
    ax.set_xlim([-1, 1])
    ax.set_yticks(yticks[id], yticks[id], fontsize=17)
    ax.set_ylim([min(yticks[id]), max(yticks[id])])
    ax.set_ylabel('') #original is 'Density'
    ax.axvline(0.3, color='gray',  linestyle=':')
    ax.axvline(-0.3, color='gray',  linestyle=':')

handles, labels = axs.flatten().tolist()[0].get_legend_handles_labels()
fig.legend(handles=handles[0:9], loc=(0.275,0.005), ncol=4, fontsize=20) #loc=(0.855,0.035)
fig.tight_layout()
plt.subplots_adjust(bottom=0.08, wspace=0.3, hspace=0.4)
# plt.savefig('hist_pcc_3k_individual_pub_supp_4_new_edited.png', dpi=300)
# plt.savefig('hist_pcc_23k_individual_pub_supp_4_new_edited_large_test.svg', dpi=600)
plt.show()

### 4.2.5 Correlation of Correlation

In [ ]:
# To evaluate how similar is Data-23K to Data-Whole and Data-3K
labels = ['GC Content', 'Gibbs Free Energy', 'Minimum Free Energy','Nucleotide A','Nucleotide U','Nucleotide G','Nucleotide C','Melting Temperature', 'AU Skew', 'GC Skew', 'Shannon Entropy', 'U at Position 1','G at Position 1',  'GG across Sequence', 'UA across Sequence', 'CC across Sequence', 'GC across Sequence', 'UU across Sequence']

corr_df = pd.DataFrame(index=np.arange(0, len(labels)*3), columns=['Interpretable Features', 'Datasets','PCC', 'P-Value (PCC)']) #18*3

In [ ]:
# PCC 
corr_whole = [gc_corr, gfe_corr, mfe_corr,a_corr, u_corr, g_corr, c_corr, tm_corr, au_skew_corr, gc_skew_corr, shannon_corr, u1_corr,  g1_corr, ggall_corr, uaall_corr, ccall_corr, gcall_corr, uuall_corr] 

corr_23 = [gc_corr_23, gfe_corr_23, mfe_corr_23, a_corr_23, u_corr_23, g_corr_23, c_corr_23, tm_corr_23, au_skew_corr_23,gc_skew_corr_23, shannon_corr_23, u1_corr_23,  g1_corr_23, ggall_corr_23, uaall_corr_23, ccall_corr_23, gcall_corr_23, uuall_corr_23] 

corr_3 = [gc_corr_3, gfe_corr_3, mfe_corr_3, a_corr_3, u_corr_3, g_corr_3, c_corr_3, tm_corr_3, au_skew_corr_3,gc_skew_corr_3, shannon_corr_3,  u1_corr_3,  g1_corr_3, ggall_corr_3, uaall_corr_3, ccall_corr_3, gcall_corr_3, uuall_corr_3] 

for z in range(len(labels)):
    corr_df.loc[z] = [labels[z], 'Data-Whole vs. Data-23K', pearsonr(corr_whole[z], corr_23[z])[0], pearsonr(corr_whole[z], corr_23[z])[1], pearsonr(corr_whole[z],corr_23[z])[0], pearsonr(corr_whole[z],corr_23[z])[1]] #'Data-Whole vs. Data-23K'

    corr_df.loc[z+18] = [labels[z], 'Data-Whole vs. Data-3K', pearsonr(corr_whole[z], corr_3[z])[0], pearsonr(corr_whole[z], corr_3[z])[1], pearsonr(corr_whole[z], corr_3[z])[0], pearsonr(corr_whole[z], corr_3[z])[1]]

    corr_df.loc[z+18*2] = [labels[z], 'Data-3K vs. Data-23K', pearsonr(corr_3[z], corr_23[z])[0], pearsonr(corr_3[z], corr_23[z])[1], pearsonr(corr_3[z], corr_23[z])[0], pearsonr(corr_3[z], corr_23[z])[1]] #'Data-3K vs. Data-23K'


In [ ]:
corr_df = corr_df.set_index(['Interpretable Features'])
corr_df = corr_df.loc[labels]
corr_df = corr_df.reset_index()
for x in corr_df.columns[2:]:
    corr_df[x] = np.float64(corr_df[x])

# corr_df.to_csv('corr_of_pcc_df_all.csv')
corr_df

In [ ]:
# Bar Chart to show the correlation of correlation 
labels = ['GC Content', 'Gibbs Free Energy', 'Minimum Free Energy','Nucleotide A','Nucleotide U','Nucleotide G','Nucleotide C',  'Melting Temperature', 'AU Skew', 'GC Skew', 'Shannon Entropy', 'U at Position 1','G at Position 1', 'GG across Sequence', 'UA across Sequence', 'CC across Sequence', 'GC across Sequence', 'UU across Sequence']

# cmap = plt.get_cmap("Set1")
# colors = [cmap(i/9) for i in range(9)]
colors = ['#e0f3db', '#a8ddb5', '#43a2ca']

fig, ax = plt.subplots(1,1, figsize=(14,7))

for id in range(len(labels)):
    p = ax.bar(id+0.25*-1, pd.to_numeric(corr_df['PCC'][id*3]).round(5), color=colors[0], label='Data-Whole vs. Data-23K', width=0.25)
    p1 = ax.bar(id+0.25*0, pd.to_numeric(corr_df['PCC'][id*3+1]).round(5), color=colors[1], label='Data-Whole vs. Data-3K', width=0.25)
    p2 = ax.bar(id+0.25*1, pd.to_numeric(corr_df['PCC'][id*3+2]).round(5), color=colors[2],label='Data-23K vs. Data-3K', width=0.25)
    ax.bar_label(p, label_type='center',color='black',fontsize=12, rotation=90)
    ax.bar_label(p1, label_type='center', color='black',fontsize=12, rotation=90)
    ax.bar_label(p2, label_type='center', color='black',fontsize=12, rotation=90)
    # ax.set_title(labels[id])
plt.xticks(rotation=90, fontsize=12)
plt.yticks(fontsize=12)
ax.set_ylim(0.5, 1.03)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels([corr_df['Interpretable Features'][i*3] for i in range(18)])


plt.xlim(-0.75,len(corr_df[0:18])-0.25)
# fig.suptitle("PCC of Each RNA Type and Dataset", fontsize=18)
handles, labels = ax.get_legend_handles_labels()
plt.subplots_adjust(top=0.94)
fig.legend(handles=[handles[0],handles[10],handles[29]],labels=[labels[0],labels[10],labels[29]],loc=(0.39,0.92),ncols=3,fontsize=12)
fig.tight_layout()

# plt.savefig('pcc_of_pcc_bar_wopal_edited.png', dpi=300)
# plt.savefig('pcc_of_pcc_bar_wopal_edited.svg', dpi=600)
plt.show()